# PT2: 3D Launch Geometry and Driver Pipeline

The goal of this notebook is to unpack how our 4D tensors are mapped to hardware threads on the GPU. How launch geometry is calculated and cached, and how drivers manage memory pass throughs. We've already shown how much faster the custom kernels are, but we still have to peel back all the layers to understand why it works so fast. 
If you have been reading along and haven't already, I recommend still skimming through the two files, that way it's a little easier to follow along in the notebooks. 

<img src="../../../notebooks/assets/max_pooling_cfd.png" width="1200" alt="Max Pooling CFD Flowchart">


### Mapping our Tensors to 3D Thread Grids via (`_get_launch_geometry`): 

The flowchart is a great reminder in knowing how `Layer._compile_for_device` works. In this case it's for max pooling as its slightly more complicated then `AvgPool2d` (because of the divergance between inferance and training). 


Now, we'd like to explore what happens when we take the forward and backward route of the gpu layers. So lets dive in!
While its a bit wiser to explain how we map the kernels, I'd rather wait until later on in the notebook, so we'll discuss the main helper methods inside class `_PoolNd` first. 

Our normal `self.forward` layer is replaced with one of the two dynamic methods at runtime.

In [2]:
import numpy as np
class MaxPool2d_ex(): # base class Layer omitted
    def _forward_gpu(self, inputs, training, block_z):
        kernel = self._kernel_train if training else self._kernel_infer
        return self._forward_gpu_common(
            inputs,
            pad_value=-np.inf,
            block_z=block_z,
            kernel=kernel,
            track_indices=training,
        )

    def _backward_gpu(self, dvalues, block_z):
        return self._backward_gpu_common(
            dvalues,
            block_z,
            kernel=self._kernel_backward_nonoverlap,
            aux_args=(self.max_indices,),
        )

The forward and backward passes both pass in similar arguments, lets start with the forward pass: 
* `inputs`: Our input tensor we perform the calculation on.
* `pad_value`: we hardcode `-np.inf` as noted in the 1st notebook, average pooling would require `0.0`. 
* `block_z`: Either a block depth of 2 or 4 depending on backend type
* `kernel`: Our custom kernel that is compiled once
* `track_indices`: Given the training flag (will likely be deprected) if true will keep a 4D tensor that tracks the `self.max_indices` for backpropagation. 

The backward pass will pass in: 
`aux_args`: This is needed to compute the location of the winners from the forward pass, then we'll route 100% of the gradient directly to the max positon. 

Both average and max pooling classes will then go inside `_PoolNd._forward_gpu_common`. This layer will handle the forward input `inputs.padded`, setup launch geometry `_PoolNd._get_launch_geometry`, handle the max vs inference logic for max pooling, and call our custom kernel. The first notebook already went over `self._prepare_forward_input` so we'll go inside `_PoolNd._get_launch_geometry` instead. 

In [ ]:
class _PoolNd():
    def __init__(self):
        self._launch_cache = {}

    def _get_launch_geometry(self, S, H_pad, W_pad, C, H_out, W_out, block_z):

        key = (S, H_pad, W_pad, C, H_out, W_out, block_z)
        cached = self._launch_cache.get(key)
        if cached is not None:
            return cached

        fH, fW = self.filter_size
        sH, sW = self.stride

        block_x = min(32, C)
        block_y = 8
        block_dim = (block_x, block_y, block_z)
 
        grid_x = (C + block_x - 1) // block_x
        grid_y = (W_out + block_y - 1) // block_y
        grid_z = (H_out * S + block_z - 1) // block_z
        grid_dim = (grid_x, grid_y, grid_z)
 
        static_args = (
            np.int32(S), np.int32(H_pad), np.int32(W_pad), np.int32(C),
            np.int32(fH), np.int32(fW), np.int32(sH), np.int32(sW),
            np.int32(H_out), np.int32(W_out),
        )
 
        result = (block_dim, grid_dim, static_args)
        self._launch_cache[key] = result
        return result
    
    def _forward_gpu_common(self, inputs, pad_value, block_z, kernel, track_indices=False):
        xp, inputs_padded, S, H_out, W_out = self._prepare_forward_input(inputs, pad_value=pad_value)

        self.inputs_shape = inputs.shape
        self.padded_shape = inputs_padded.shape
        _, H_pad, W_pad, C = inputs_padded.shape

        self.output = xp.empty((S, H_out, W_out, C), dtype=inputs.dtype)
        block_dim, grid_dim, static_args = self._get_launch_geometry(
            S, H_pad, W_pad, C, H_out, W_out, block_z
        )

        if track_indices:
            self.max_indices = xp.empty((S, H_out, W_out, C), dtype=xp.int32)
            aux_args = (self.max_indices,)
        else:
            self.max_indices = None
            aux_args = ()

        kernel_args = (inputs_padded, self.output) + aux_args + static_args
        kernel(grid_dim, block_dim, kernel_args)

        return self.output

### How does `self._get_launch_geometry` work and why is it a $O(1)$ Lookup? 

We'll use a dictionary for this part, our key becomes the input parameters, and we'll check if we already cached the result. If not, we'll setup the actual grid dimensions, blocks, and static arguments needed for our kernel. These are needed for when we call the kernel later. 
```python
key = (S, H_pad, W_pad, C, H_out, W_out, block_z)
cached = self._launch_cache.get(key)
if cached is not None:
    return cached #this contains the tuple (block_dim, grid_dim, static_args)
# Used in static_args
fH, fW = self.filter_size
sH, sW = self.stride

block_x = min(32, C)
block_y = 8
#block_z = 2 or 4
block_dim = (block_x, block_y, block_z)
```

Since we use a dictionary, we can do a simple $O(1)$ lookup for the key, with the value from the key being a tuple of arguments that we'll pass into a kernel. 
As a quick reminder, memory transactions are done usually through 32 threads per transaction, meaning we'll conform our block size to be divisible by 32 (512 for HIP, 1024 for CUDA). 

Hardware & Execution Context (GPU Architecture Alignment)
* `block_x` **& Warp Coalescing**: Setting block_x = min(32, C) directly maps the inner dimension to a single 32-thread hardware Warp (NVIDIA) or Wavefront (AMD). When $C \ge 32$, all 32 threads in the warp execute together in lockstep, merging 32 individual memory accesses into a single 128-byte coalesced memory transaction over the VRAM bus.

* `block_z` **& L1 Cache / Occupancy Tuning**: block_z assigns 3D depth (channels/batches) to adjacent threads within the same physical compute core (SM or CU).
    * L1 Cache Reuse: Neighboring $Z$-threads process adjacent channels/batches simultaneously, allowing them to hit the same local L1 cache lines with minimal latency.

* **Latency Hiding via SM Warp Scheduling**:

Memory transactions from VRAM to cache take **200-400** clock cycles (modern consumer GPUs sit around 2.7-3.2 GHz). GPUs hide this latency by maintaining thousands of active thread states directly on-chip and switching execution instantly when a memory fetch stalls a thread group.

* **NVIDIA (Streaming Multiprocessor - SM)**: 
    * **State Allocation**: Up to **64 active Warps (2,048 threads)** stay resident in the SM's **64K 32-bit Register File.**
    * **Scheduling**: Four independent **Warp Schedulers** check resident warps every clock cycle. When Warp 0 stalls on a VRAM fetch, the schedular instantly swaps executions to an eligible Warp on the next clock edge with **0-cycle overhead**. 
* **AMD (RDNA Archtiecture - CU/WGP)**:
    * **State Allocation**: Up to **40 Wavefronts** stay resident per Compute Unit (CU) allocated across the physical **Vector General Purpose Register (VGPR)** pool 
    * **Four Wavefront Schedulers per CU** manage SIMD execution vector units. If Wavefront 0 stalls on a global BUFFER_LOAD, the hardware scheduler immediately hands the SIMD unit to Wavefront 1 with **0-cycle latency**, leveraging on-chip VGPR state retention.
    > **Architectural Note (RDNA 4 Dynamic VGPRs) [^1]:** > Modern AMD architectures (RDNA 4+) introduce *Dynamic VGPR Allocation*. Rather than statically reserving peak register space for a Wavefront's full lifecycle, the hardware dynamically allocates and frees 1024-bit vector registers on the fly based on runtime demands. This drastically lowers register pressure (a power user can now set block_z = 4), allowing higher resident Wavefront occupancy on the CU for improved latency hiding.

    [^1]: Chips and Cheese: [Dynamic Register Allocation on AMD's RDNA 4 Architecture](https://chipsandcheese.com/p/dynamic-register-allocation-on-amds)

At a higher level, our $Z$-axis enables our latency hiding (as shown above), avoiding the VRAM transaction and means that we can perform the pooling kernels on multiple threads. 

### HOW DOES THE GRID... GRID???

Right after we define how threads form the Warp or Wavefront inside a single thread block, we have to observe how thread blocks are distributed across the entire dataset using a Grid. For this, our fundamental target is **1-to-1 work mapping**, meaning every individual element in the output tensor must be owned and computed by exactly one GPU thread. 

$$N_{\text{total\_work}} = S \times H_{out} \times W_{out} \times C$$

For this, we have to solve two different problems: 
1. **Dimension Compression**: How to map a 4D output tensor onto a maximum 3D GPU Grid $(X, Y, Z)$.
2. **Coverage Arithmetic**: How to calculate the minimum integer number of blocks needed along each axis so no output elements are left out.

```python
grid_x = (C + block_x - 1) // block_x
grid_y = (W_out + block_y - 1) // block_y
grid_z = (H_out * S + block_z - 1) // block_z
grid_dim = (grid_x, grid_y, grid_z)
```

### Dimension Compression Via Grid Dimensions. 

GRID DIMENSIONS **(grid_x, grid_y, grid_z)**  
  grid_x ───────┤ Channels (C)                            
  grid_y ───────┤ Output Width (W_out)                    
  grid_z ───────┤ Output Height × Batch (H_out × S)       

`grid_x`: We're changing the channels $C$ across blocks along the $X$-axis. 
  * If $C=64$ and `block_x=32`, `grid_x = 2`. Block 0 will handle the first 32 input channels (0...31), with Block 1 handling the channels (32...64). 
  * Because $X$ is the inner contiguous dimension in C-contiguous memory layouts, each `grid_x` block generates aligned 128-byte memory transactions across the 32 threads.  

`grid_y`: Where we span the horizontal spatial dimension of the output feature map.
  * As `block_y = 8`, `grid_y` determines how many vertical slices are needed to cover the entire output width $W_{out}$.  

`grid_z`: This is where we collapse the batch size $S$ and output height $H_{out}$ into a single composite $Z$-axis.
  * Because we fused both dimensions into the $Z$-axis, the kernel upacks the sample index and height row on the fly.
    * **Note**: It does me we perform an extra operation on the core itself, but since we're memory bound this doesn't affect performance. 
    
  $$
  \text{grid\_z} = \frac{(H_{out} \times S + \text{block\_z} - 1)}{\text{block\_z}}
  $$

Because $H_{out}$ and $S$ are fused into `grid_z`; the CUDA/HIP C++ kernel unpacks the sample index and height row on the fly in hardware registers:

### Why This Grid Architecture? (Hardware Limits & Thread Safety)

You might wonder: *Why not fuse height and width into `grid_y` and dedicate `grid_z` solely to the batch size $S$?* 

Two critical hardware constraints and execution mechanics dictate why the grid is structured this way:

1. **Hard Hardware Grid Limits (`gridDim.y` $\le$ 65,535):**
   GPU architectures strictly cap the $Y$ and $Z$ grid dimensions at **65,535 blocks**. If we fused $H_{out} \times W_{out}$ onto `grid_y` (e.g., a high-resolution $2048 \times 2048$ image with `block_y = 32`), the required grid size would be:
   $$\text{grid\_y} = \frac{2048 \times 2048}{32} = 131,072 \quad (\text{Exceeds our 65,535 limit})$$
   This would trigger an `invalid configuration argument` crash. Fusing $S \times H_{out}$ into `grid_z` and keeping $W_{out}$ on `grid_y` prevents either spatial axis from violating this hardware ceiling.

2. **Guaranteed Single-Sample Isolation (No Cross-Sample Contamination):**
   Even though $S$ and $H_{out}$ are fused into a single composite $Z$-axis, **a single thread never crosses sample boundaries**. The kernel unpacks coordinates on the fly:
   ```cpp
   int h_s   = blockIdx.z * blockDim.z + threadIdx.z;
   int h_out = h_s % H_out;  // Spatial row
   int s     = h_s / H_out;  // Batch sample index

Each thread calculates a fixed scalar $s$, computes its reduction window for that specific sample only, and exits. Furthermore, because block_x (Channels $C$) maps to the inner $X$-axis, adjacent threads in a warp share identical $Y$ and $Z$ coordinates—meaning warps never mix samples, preserving 128-byte memory coalescing.


## Coverage Arithmetic: Why the Equations Are Built This Way:

We want to cover the entire size $N$ using blocks of size $B$, given the required number of blocks $G$ we must satisfy: 
$$
G \times B \ge N
$$

The code is given $B$ through the block variables, and $N$ through the shape of the tensor; solving for $B$ 
$$
G \le \frac{N}{B}\quad\quad \text{or} \quad\quad G = \lceil \frac{N}{B}\rceil
$$
While the left equation is true, we cannot allocate fractional blocks, so we'll use a ceiling operation. Integer calculations like this are best suited for the CPU. To implement this without floating-point casts, we add ($B - 1$) to the numerator before integer division:

$$\left\lceil \frac{N}{B} \right\rceil = \left\lfloor \frac{N + B - 1}{B} \right\rfloor = (N + B - 1) // B$$

Proof:
* Case 1: $N$ is an exact multiple of $B$ ($N = k \cdot B$)
$$\left\lfloor \frac{k \cdot B + B - 1}{B} \right\rfloor = \left\lfloor \frac{(k+1)B - 1}{B} \right\rfloor =  \left\lfloor (k + 1) - \frac{1}{B} \right\rfloor = k
$$
  * **Result**: We allocate $k$ blocks to the input tensor, with zero blocks wasted!
* Case 2: $N$ is NOT a multiple of $B$ ($N = k \cdot B + r$, where $1 \le r \le B - 1$)
$$\left\lfloor \frac{(k \cdot B + r) + B - 1}{B} \right\rfloor = \left\lfloor \frac{(k+1)B + r - 1}{B} \right\rfloor =  \left\lfloor (k + 1) - \frac{r - 1}{B} \right\rfloor = (k+1)
$$
```C++
// 1. Thread calculates its flattened global Z index:
int h_s = blockIdx.z * blockDim.z + threadIdx.z;
  * **Result**: We allocate $k+1$ blocks to the input tensor, with one block partially wasted. 
// 2. Index Unraveling:
int h_out = h_s % H_out;  // Remainder = Spatial Row inside the image (Height)
int s     = h_s / H_out;  // Quotient  = Batch Sample Index
```
#### Deriving the Grid Axes Now

We've proven that it works for both cases, now we can assign them to our 3D grid. 

1. Channel Axis (`grid_x`):
$$\text{grid\_x} = \left\lceil \frac{C}{\text{block\_x}} \right\rceil = (C + \text{block\_x} - 1) // \text{block\_x}$$
2. Spatial Width Axis (`grid_y`)
$$\text{grid\_y} = \left\lceil \frac{W_{out}}{\text{block\_y}} \right\rceil = (W_{out} + \text{block\_y} - 1) // \text{block\_y}$$
3. Flattened Spatial Height and Batch Samples Axis (`grid_z`)
$$\text{grid\_z} = \left\lceil \frac{H_{out} \times S}{\text{block\_z}} \right\rceil = (H_{out} \cdot S + \text{block\_z} - 1) // \text{block\_z}$$

With the same forumla being applied to all three axes, we cover the remainder for chanels, our width axis, and the last axis remains fully saturated. For `grid_z`, we allocate a *remainder* block because the last few rows of the final batch sample spill over into a new block. 

```python
static_args = (
    np.int32(S), np.int32(H_pad), np.int32(W_pad), np.int32(C),
    np.int32(fH), np.int32(fW), np.int32(sH), np.int32(sW),
    np.int32(H_out), np.int32(W_out),
)

result = (block_dim, grid_dim, static_args)
self._launch_cache[key] = result
return result
```

The `static_args` tuple is not complicated at all, we force cast our integer sizes for the kernel when it reads this later, and each input element is needed to calculate the actual output tensor, which is why they are all included.

Finally, the last line will save a second, larger tuple as the value for the `self._launch_cache`, and we'll return this result anyways.

# A Small Intro into `pooling_kernel.py`

The kernel file includes three major things, with the first two being created using dictionaries and Template:
1. Template Strings: These are our base configuration for the pooling layer, while we could do away with this approach since both `AvgPool2d` and `MaxPool2d` benefit from using the same template for the forward pass. 
2. OP Strings: These strings include the specialized action for the kernel. 
    * EX `_AVG_OP`: T
    ```C++
    _AVG_OP = {
    "name": "avg",
    "aux_out_decl": "",
    "reduce_init": "float sum_val = 0.0f;",
    "reduce_body": "sum_val += x[in_idx];",
    "writeback": "out[out_idx] = sum_val / (float)(fH * fW);",
    }
    ```
    Each key has a name string, and each value includes the specific information corresponding to the string. If we need multiple lines per key, we can add it following standard C++ syntax, and using `\n` in python

At the bottom of the file, we have two helper functions that create the kernel, and four `get` methods that call for the respective kernel. This approach technically could be optimized a little bit, but since calling a function is only a few nanoseconds of overhead this is really a non-issue.

```python
def get_max_pool2d_forward_kernel(variant: str, training: bool = True):
    """Returns a compiled RawKernel for the given variant ('cuda' or 'hip').
    Memoized via _pool_kernel_cache — no lazy self-rewriting needed.
    If we're not training, remove the 4D max_indicies tensor using a similar
    kernel _MAX_OP_INFERENCE
    """
    op = _MAX_OP if training else _MAX_OP_INFERENCE
    return _get_compiled_forward_kernel(op, variant)

def get_max_pool2d_backward_kernel(variant: str):
    """Returns a compiled RawKernel implementing the MaxPool2d backward
    scatter for the non-overlapping-window case (filter_size == stride
    only). Every padded-input position is the argmax target of at most
    one output window in this regime, so the kernel does a direct write
    instead of an atomicAdd. Do NOT use this for overlapping windows —
    that requires the atomic/scatter-add fallback path instead.
    """
    return _get_compiled_max_backward_kernel(variant)

# Helper Functions for AvgPool2d
def get_avg_pool2d_forward_kernel(variant: str):
    """Returns a compiled RawKernel for the given variant ('cuda' or 'hip').
    Memoized via _pool_kernel_cache — no lazy self-rewriting needed.
    """
    return _get_compiled_forward_kernel(_AVG_OP, variant)

def get_avg_pool2d_backward_kernel(variant: str):
    """Returns a compiled RawKernel implementing the AvgPool2d backward
    scatter for the non-overlapping-window case (filter_size == stride
    only). Do NOT use this for overlapping windows —
    that requires the atomic/scatter-add fallback path instead.
    """
    return _get_compiled_avg_backward_kernel(variant)
```

All four methods will call their respective compiled kernel. The `variant` string is either `hip` or `cuda` depending on the users GPU. This is because writing `cupy.RawKernel`s requires rewriting the code based on the GPU backend.
A flow chart highlighting how the `pooling_kernel.py` file is included below, where we'll go much more in depth in the third and final notebook.   

<img src="../../../notebooks/assets/pooling_kernel_cfd.png" width="1500" alt="Max Pooling CFD Flowchart">